# T-Series Forecasting — Silver Layer
## Cleaned Delta tables + MERGE + feature engineering

**Catalog:** `workspace`  
**Silver schema:** `workspace.silver`

This notebook reads Bronze Delta tables, standardizes timestamps/types, removes invalid duplicates, creates useful feature tables, and writes them with **Delta MERGE**.

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

In [0]:
telemetry = spark.table(f"{BRONZE}.telemetry")
errors = spark.table(f"{BRONZE}.errors")
failures = spark.table(f"{BRONZE}.failures")
maint = spark.table(f"{BRONZE}.maint")
machines = spark.table(f"{BRONZE}.machines")

display(telemetry.limit(5))
display(machines.limit(5))

## 1. Clean and standardize source tables

In [0]:
# Databricks inferSchema normally identifies these timestamps correctly.
# to_timestamp also safely handles string input.
telemetry_s = (
    telemetry
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .filter(F.col("machineID").isNotNull() & F.col("datetime").isNotNull())
    .dropDuplicates(["machineID","datetime"])
)

errors_s = (
    errors
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .filter(F.col("machineID").isNotNull() & F.col("datetime").isNotNull())
    .dropDuplicates(["machineID","datetime","errorID"])
)

failures_s = (
    failures
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .filter(F.col("machineID").isNotNull() & F.col("datetime").isNotNull())
    .dropDuplicates(["machineID","datetime","failure"])
)

maint_s = (
    maint
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .filter(F.col("machineID").isNotNull() & F.col("datetime").isNotNull())
    .dropDuplicates(["machineID","datetime","comp"])
)

machines_s = (
    machines
    .withColumn("machineID", F.col("machineID").cast("int"))
    .dropDuplicates(["machineID"])
)

for name, df in {
    "telemetry_clean": telemetry_s,
    "errors_clean": errors_s,
    "failures_clean": failures_s,
    "maint_clean": maint_s,
    "machines_clean": machines_s
}.items():
    print(name, df.count())

## 2. Silver MERGE helper

In [0]:
def merge_to_silver(df, table_name, keys):
    full = f"{SILVER}.{table_name}"
    if not spark.catalog.tableExists(full):
        df.write.format("delta").mode("overwrite").saveAsTable(full)
        print("Created:", full)
        return

    target = DeltaTable.forName(spark, full)
    condition = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])

    (
        target.alias("t")
        .merge(df.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGED:", full)

In [0]:
merge_to_silver(telemetry_s, "telemetry_clean", ["machineID","datetime"])
merge_to_silver(errors_s, "errors_clean", ["machineID","datetime","errorID"])
merge_to_silver(failures_s, "failures_clean", ["machineID","datetime","failure"])
merge_to_silver(maint_s, "maint_clean", ["machineID","datetime","comp"])
merge_to_silver(machines_s, "machines_clean", ["machineID"])

## 3. Create analytical feature tables

In [0]:
# Latest machine/component maintenance timestamp available at each telemetry point.
maint_last = (
    maint_s.select("machineID","datetime","comp")
    .withColumnRenamed("datetime","maint_datetime")
)

w = Window.partitionBy("machineID").orderBy("maint_datetime").rowsBetween(Window.unboundedPreceding, 0)

telemetry_features = (
    telemetry_s
    .withColumn("date", F.to_date("datetime"))
)

# Machine age proxy from first observed telemetry date.
first_ts = telemetry_s.groupBy("machineID").agg(F.min("datetime").alias("machine_first_seen"))
telemetry_features = (
    telemetry_features.join(first_ts, "machineID", "left")
    .withColumn("machine_age_days",
                F.datediff(F.to_date("datetime"), F.to_date("machine_first_seen")))
)

In [0]:
# Add model information.
telemetry_features = telemetry_features.join(
    machines_s.select("machineID", "model"), "machineID", "left"
)

# Rolling sensor statistics useful for Q3/Q5/Q6/Q8.
sensor_cols = [c for c in ["volt","rotate","pressure","vibration"] if c in telemetry_features.columns]
w24 = Window.partitionBy("machineID").orderBy(F.col("datetime").cast("long")).rangeBetween(-24*3600, 0)

for c in sensor_cols:
    telemetry_features = (
        telemetry_features
        .withColumn(f"{c}_rolling_mean_24h", F.avg(c).over(w24))
        .withColumn(f"{c}_rolling_std_24h", F.stddev(c).over(w24))
    )

In [0]:
# Days since latest maintenance event for each machine.
m = maint_s.select("machineID","datetime").withColumnRenamed("datetime","maint_ts")
joined = telemetry_features.join(
    m,
    (telemetry_features.machineID == m.machineID) &
    (m.maint_ts <= telemetry_features.datetime),
    "left"
)
wm = Window.partitionBy(
    telemetry_features.machineID, telemetry_features.datetime
).orderBy(F.col("maint_ts").desc())

telemetry_features = (
    joined.withColumn("_rn", F.row_number().over(wm))
    .filter("_rn = 1")
    .drop("_rn", "maint_ts")
    .withColumn(
        "days_since_maintenance",
        F.datediff(F.to_date("datetime"), F.to_date("maint_ts"))
    )
    if False else telemetry_features
)

The previous block intentionally avoids a potentially expensive range join for large telemetry data. We instead create a reusable maintenance timeline with a window-based as-of join below.

In [0]:
# Efficient as-of maintenance date using union + forward fill.
maint_dates = maint_s.select("machineID","datetime").withColumn("event_type", F.lit("maintenance"))
telemetry_markers = telemetry_s.select("machineID","datetime").withColumn("event_type", F.lit("telemetry"))

events = telemetry_markers.unionByName(maint_dates)
wfill = Window.partitionBy("machineID").orderBy("datetime").rowsBetween(Window.unboundedPreceding, 0)

asof = (
    events
    .withColumn("last_maintenance",
                F.last(F.when(F.col("event_type")=="maintenance", F.col("datetime")),
                       ignorenulls=True).over(wfill))
    .filter(F.col("event_type")=="telemetry")
    .select("machineID","datetime","last_maintenance")
)

telemetry_features = (
    telemetry_features.drop("days_since_maintenance")
    .join(asof, ["machineID","datetime"], "left")
    .withColumn(
        "days_since_maintenance",
        F.datediff(F.to_date("datetime"), F.to_date("last_maintenance"))
    )
)

## 4. Failure-risk training dataset

In [0]:
# Label: a failure occurring in the next 24 hours for the same machine.
fail = failures_s.select(
    "machineID",
    F.col("datetime").alias("failure_ts")
)

risk_base = telemetry_features.alias("t").join(
    fail.alias("f"),
    (F.col("t.machineID") == F.col("f.machineID")) &
    (F.col("f.failure_ts") > F.col("t.datetime")) &
    (F.col("f.failure_ts") <= F.col("t.datetime") + F.expr("INTERVAL 24 HOURS")),
    "left"
)

risk = (
    risk_base
    .groupBy([F.col(f"t.{c}") for c in telemetry_features.columns])
    .agg(F.max(F.when(F.col("f.failure_ts").isNotNull(), 1).otherwise(0)).alias("failure_next_24h"))
)

In [0]:
def merge_feature(df, table_name, keys):
    full = f"{SILVER}.{table_name}"
    if not spark.catalog.tableExists(full):
        df.write.format("delta").mode("overwrite").saveAsTable(full)
    else:
        DeltaTable.forName(spark, full).alias("t").merge(
            df.alias("s"),
            " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("MERGED:", full)

merge_feature(telemetry_features, "telemetry_features", ["machineID","datetime"])
merge_feature(risk, "failure_risk_features", ["machineID","datetime"])

In [0]:
for t in ["telemetry_clean","errors_clean","failures_clean","maint_clean","machines_clean",
          "telemetry_features","failure_risk_features"]:
    print(f"{SILVER}.{t}")
    display(spark.table(f"{SILVER}.{t}").limit(3))